# Web Scraping 101
## STARS Computing Corps
Build a dataset of fictional tech events. Run cells from top to bottom. Basic Python helps, but watching and making predictions also counts.

**You will:** fetch HTML, locate repeated elements, extract fields, and export a CSV.

**Before starting:** In Colab, save your own copy if you want to retain edits. This notebook includes its own backup HTML. Blank `PAGE_URL` means backup practice, not live scraping.

## 1. Tools
Requests fetches the page. Beautiful Soup reads its HTML. pandas displays and saves rows. Run this cell while the presenter introduces the workshop.

In [ ]:
# Install only packages missing from this notebook environment.
import importlib.util
import subprocess
import sys
packages = {"requests": "requests>=2.31,<3", "bs4": "beautifulsoup4>=4.12,<5", "pandas": "pandas>=2.0,<4"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import requests
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display
print("Ready!")

## 2. Practice-page backup
Run this cell once. It stores a copy of the same fictional page. You do not need to read the long HTML string.

In [ ]:
BACKUP_HTML = '<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>STARS Workshop Practice Events</title>\n<style>body{margin:0;background:#f3f5f8;color:#162336;font:18px/1.55 system-ui,sans-serif}header,main,footer{max-width:1080px;margin:auto;padding:32px}header{padding-top:56px}h1{font-size:48px;line-height:1.1;margin:15px 0}header p{max-width:750px}.label{font-weight:700;color:#365b9d;letter-spacing:.08em}main{display:grid;grid-template-columns:repeat(auto-fit,minmax(270px,1fr));gap:20px}.event{background:white;border:1px solid #d6dce5;border-radius:12px;padding:24px}.event h2{font-size:25px;margin:0 0 18px}.event p{margin:5px 0}.format{color:#3155a0;font-weight:700}.notice{border-left:4px solid #365b9d;padding-left:16px}footer{font-size:15px}</style></head>\n<body><header><div class="label">STARS COMPUTING CORPS</div><h1>Find your next tech event</h1><p>A practice page for Web Scraping 101. Which events would you add to your calendar?</p><p class="notice"><strong>Fictional listings for learning only.</strong> These are not real events or registration opportunities. This page is designed for workshop scraping.</p></header><main>\n<article class="event">\n  <h2 class="title">Intro to Python</h2>\n  <p class="format">Online</p>\n  <p>Cost: $<span class="cost">0</span></p>\n  <p class="level">Beginner</p>\n</article>\n<article class="event">\n  <h2 class="title">Accessible Web Design</h2>\n  <p class="format">Online</p>\n  <p>Cost: $<span class="cost">0</span></p>\n  <p class="level">Beginner</p>\n</article>\n<article class="event">\n  <h2 class="title">Build Your First Robot</h2>\n  <p class="format">In person</p>\n  <p>Cost: $<span class="cost">15</span></p>\n  <p class="level">Beginner</p>\n</article>\n<article class="event">\n  <h2 class="title">Git Together</h2>\n  <p class="format">In person</p>\n  <p>Cost: $<span class="cost">0</span></p>\n  <p class="level">Beginner</p>\n</article>\n<article class="event">\n  <h2 class="title">Data for Good</h2>\n  <p class="format">Online</p>\n  <p>Cost: $<span class="cost">0</span></p>\n  <p class="level">Intermediate</p>\n</article>\n<article class="event">\n  <h2 class="title">Cloud App Lab</h2>\n  <p class="format">Online</p>\n  <p>Cost: $<span class="cost">10</span></p>\n  <p class="level">Intermediate</p>\n</article>\n<article class="event">\n  <h2 class="title">Community Coding Day</h2>\n  <p class="format">In person</p>\n  <p>Cost: $<span class="cost">0</span></p>\n  <p class="level">Beginner</p>\n</article>\n<article class="event">\n  <h2 class="title">AI Study Jam</h2>\n  <p class="format">Online</p>\n  <p>Cost: $<span class="cost">0</span></p>\n  <p class="level">Beginner</p>\n</article>\n<article class="event">\n  <h2 class="title">Hardware Hack Night</h2>\n  <p class="format">In person</p>\n  <p>Cost: $<span class="cost">20</span></p>\n  <p class="level">Intermediate</p>\n</article>\n<article class="event">\n  <h2 class="title">Career Portfolio Clinic</h2>\n  <p class="format">Online</p>\n  <p>Cost: $<span class="cost">0</span></p>\n  <p class="level">Beginner</p>\n</article>\n</main><footer>Workshop exercise: collect titles, formats, costs, and levels. Find free online events. Please fetch this practice page once per run.</footer></body></html>'

## 3. Fetch the HTML
A GET request asks the server for the page. The status check catches failed requests. A timeout prevents indefinite waiting.

**Checkpoint:** Read the source label. Are we using a live request or the backup?

Requests does not run page JavaScript. We designed this page with the event data directly in its HTML.

In [ ]:
# The organizer can set this to the published practice page before the workshop.
PAGE_URL = ""
USE_BACKUP = False  # Set True if the live site is unavailable.

if PAGE_URL and not USE_BACKUP:
    try:
        response = requests.get(PAGE_URL, timeout=10)
        response.raise_for_status()
        html = response.text
        source = "LIVE HTTP request: " + PAGE_URL
    except requests.RequestException as error:
        print("Live request failed:", type(error).__name__)
        html = BACKUP_HTML
        source = "BACKUP: parsing bundled HTML, no live website fetched"
else:
    html = BACKUP_HTML
    source = "BACKUP: parsing bundled HTML, no live website fetched"
print(source)
print(html[:300])

## 4. Extract one event
`.event` selects elements with class `event`. `select()` returns all matches; `select_one()` returns the first match. `get_text(strip=True)` extracts text and trims surrounding whitespace.

**Predict:** Will this cell print one title or all ten?

**Checkpoint:** 10 event containers and `Intro to Python`.

In [ ]:
soup = BeautifulSoup(html, "html.parser")
cards = soup.select(".event")
if not cards:
    raise ValueError("No .event elements found. Check the URL or switch USE_BACKUP to True and rerun the fetch cell.")
print("Event containers:", len(cards))
first = cards[0]
print(first.select_one(".title").get_text(strip=True))

## 5. Extract all events
Each loop starts inside one event container so titles, formats, and costs stay matched. Each dictionary becomes one table row. Convert cost to a number so we can compare prices.

**Checkpoint:** A table of 10 events with four columns.

In [ ]:
rows = []
for card in cards:
    rows.append({
        "title": card.select_one(".title").get_text(strip=True),
        "format": card.select_one(".format").get_text(strip=True),
        "cost": float(card.select_one(".cost").get_text(strip=True)),
        "level": card.select_one(".level").get_text(strip=True),
    })
events = pd.DataFrame(rows)
display(events)

## 6. Your challenge: free online events
Change only `wanted_format` and `max_cost`. Run the cell.

**Goal:** Keep online events that cost $0. Compare your result with a neighbor.

The `&` means both conditions must be true. pandas needs parentheses around each condition.

**Watching only?** Predict how many events will remain before the presenter runs it.

In [ ]:
wanted_format = "In person"  # TODO: choose "Online"
max_cost = 20  # TODO: free means 0
selected = events[
    (events["format"] == wanted_format) &
    (events["cost"] <= max_cost)
]
display(selected)
print(f"{len(selected)} matching events")

## 7. Save your dataset
Save the complete table and your filtered results. `index=False` leaves out the extra row-number column.

In [ ]:
events.to_csv("events.csv", index=False)
selected.to_csv("selected_events.csv", index=False)
print("Saved events.csv and selected_events.csv")

### Optional Colab download
Run this cell to download your filtered CSV. Outside Colab, the CSV is in the notebook working directory.

In [ ]:
try:
    from google.colab import files
except ImportError:
    print("Open selected_events.csv in your notebook working directory.")
else:
    files.download("selected_events.csv")

## 8. Where could you take this?
Build a searchable opportunity board, a calendar, or an accessibility-focused event finder.

**Stretch exercise:** add a beginner-level condition to your filter. Then add an event to your own practice page and rerun the scraper.

**Before scraping another site:** check its terms and robots.txt, prefer a suitable API, limit requests, and avoid personal or restricted data. robots.txt is guidance for crawlers, not a grant of permission. Never bypass logins or access restrictions.

**Exit question:** If a site renames its `title` class, which selector would you update?

## References
- [Requests quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [Beautiful Soup documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [pandas CSV export](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)